In [ ]:
"""
Part 1 — Batch 3D nuclei segmentation with Cellpose v4.

Input
- TIFF hyperstacks with axes ZCYX: (Z, C, Y, X)
- ImageJ metadata containing XY pixel size and Z spacing when available

Output
- One uint16 label-mask TIFF per input image
- Mask axes: ZYX
- XY and Z calibration are copied to the output metadata

How to use
1. Edit USER SETTINGS below.
2. Run this cell.
3. Select the folder containing the ZCYX TIFF images.
4. Cellpose processes every TIFF in that folder.

Install
    pip install cellpose tifffile numpy
"""

from pathlib import Path
from tkinter import Tk, filedialog

import numpy as np
import tifffile as tiff
from cellpose import core, models


# =============================================================================
# USER SETTINGS
# =============================================================================

# ---------- Image channel ----------
HOECHST_CH = 0                  # Hoechst/DNA channel index in the ZCYX TIFF

# ---------- Cellpose ----------
CELLPROB_THRESHOLD = 0.8        # Higher values produce more conservative masks
FLOW3D_SMOOTH = 1.5             # Smoothing applied to the Cellpose 3D flow field
DIAMETER = None                 # Expected nucleus diameter in pixels; None = automatic
MIN_SIZE = 2000                 # Remove segmented objects smaller than this voxel count

# ---------- TIFF calibration fallback ----------
DEFAULT_XY_UM = 0.207           # XY pixel size used if TIFF metadata is missing
DEFAULT_Z_UM = 0.2              # Z spacing used if TIFF metadata is missing

# ---------- Output ----------
OUT_SUBFOLDER = "cellpose3d_masks_ZCYX CELLPROB_THRESHOLD 0.8"
                                # Output folder created inside the selected input folder


# =============================================================================
# INPUT / OUTPUT
# =============================================================================

def select_input_folder() -> Path:
    root = Tk()
    root.withdraw()
    folder = filedialog.askdirectory(
        title="Select folder containing ZCYX TIFF hyperstacks"
    )
    root.destroy()

    if not folder:
        raise SystemExit("No folder selected.")

    return Path(folder)


def find_tiff_files(folder: Path) -> list[Path]:
    return sorted(
        path
        for path in folder.iterdir()
        if path.suffix.lower() in {".tif", ".tiff"}
    )


def read_voxel_size_um(tif_path: Path) -> tuple[float, float]:
    """
    Return XY pixel size and Z spacing in micrometers.

    TIFF metadata are used when available. Otherwise, the fallback values
    defined in USER SETTINGS are used.
    """
    with tiff.TiffFile(tif_path) as tif:
        metadata = tif.imagej_metadata or {}

    xy_um = float(
        metadata.get("pixel_width", DEFAULT_XY_UM)
    )
    z_um = float(
        metadata.get("spacing", DEFAULT_Z_UM)
    )

    return xy_um, z_um


def save_mask(
    output_path: Path,
    masks: np.ndarray,
    xy_um: float,
    z_um: float,
) -> None:
    tiff.imwrite(
        output_path,
        masks.astype(np.uint16),
        imagej=True,
        metadata={
            "axes": "ZYX",
            "unit": "um",
            "spacing": z_um,
            "pixel_width": xy_um,
            "pixel_height": xy_um,
        },
    )


# =============================================================================
# CELLPOSE SEGMENTATION
# =============================================================================

def segment_image(
    image_path: Path,
    model: models.CellposeModel,
) -> tuple[np.ndarray, float, float]:
    image = tiff.imread(image_path)

    if image.ndim != 4:
        raise ValueError(
            f"{image_path.name}: expected ZCYX input, got shape {image.shape}."
        )

    if not 0 <= HOECHST_CH < image.shape[1]:
        raise ValueError(
            f"{image_path.name}: HOECHST_CH={HOECHST_CH} is outside the "
            f"available channel range 0-{image.shape[1] - 1}."
        )

    xy_um, z_um = read_voxel_size_um(image_path)
    anisotropy = z_um / xy_um

    nuclei = image[:, HOECHST_CH].astype(np.float32)

    masks, *_ = model.eval(
        nuclei,
        do_3D=True,
        z_axis=0,
        channel_axis=None,
        normalize=True,
        diameter=DIAMETER,
        cellprob_threshold=CELLPROB_THRESHOLD,
        min_size=MIN_SIZE,
        anisotropy=anisotropy,
        flow3D_smooth=FLOW3D_SMOOTH,
        progress=True,
    )

    return masks, xy_um, z_um


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    input_dir = select_input_folder()
    output_dir = input_dir / OUT_SUBFOLDER
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_paths = find_tiff_files(input_dir)

    if not image_paths:
        raise FileNotFoundError(
            f"No TIFF files were found in {input_dir}."
        )

    gpu_enabled = core.use_gpu()
    model = models.CellposeModel(
        gpu=gpu_enabled
    )

    print(f"Input folder: {input_dir}")
    print(f"Output folder: {output_dir}")
    print(f"GPU enabled: {gpu_enabled}")
    print(f"TIFF files found: {len(image_paths)}")

    for index, image_path in enumerate(
        image_paths,
        start=1,
    ):
        print(
            f"[{index}/{len(image_paths)}] Segmenting: "
            f"{image_path.name}"
        )

        masks, xy_um, z_um = segment_image(
            image_path,
            model,
        )

        output_path = (
            output_dir
            / f"{image_path.stem}_cellpose3d_mask.tif"
        )

        save_mask(
            output_path,
            masks,
            xy_um,
            z_um,
        )

        print(
            f"Saved: {output_path.name} "
            f"({int(masks.max())} labels)"
        )

    print("Cellpose batch segmentation completed.")


if __name__ == "__main__":
    main()

In [ ]:
"""
Part 2 — Manual mask cleaning, morphometry, z-slice intensity, ESA, and WI analysis.

Input
- ZCYX TIFF images
- Cellpose masks from Cell 1:
    *_cellpose3d_mask.tif

Output
- Cleaned masks:
    *_mask_manualclean.tif
- Per-nucleus crops:
    *_nuc###_stack_multich.tif
    *_nuc###_zproj_multich.tif
- Per-image morphometry/intensity CSV:
    *_morphometry_manualclean.csv
- WI diagnostic TIFFs
- Combined CSV:
    merged_wrinkle_morphology.csv

Workflow
1. Edit USER SETTINGS below.
2. Run this cell.
3. Select the image folder and Cellpose-mask folder.
4. Each image opens one at a time in Napari.
5. Edit the mask manually, remove labels when needed, and relabel.
6. Click "Measure, Crop, and Analyze".
7. The viewer closes and the next image opens automatically.

Install
    pip install numpy pandas scikit-image tifffile napari magicgui
"""

from itertools import product
from pathlib import Path
from tkinter import Tk, filedialog
import re

import napari
import numpy as np
import pandas as pd
import tifffile as tiff
from magicgui import magicgui
from skimage import exposure, feature, measure, transform
from skimage.morphology import disk, erosion
from skimage.transform import resize


# =============================================================================
# USER SETTINGS
# =============================================================================

# ---------- Image channels ----------
HOECHST_CH = 0                         # Hoechst/DNA channel index in original ZCYX TIFF
LAMIN_CH = 1                           # Lamin channel index in original ZCYX TIFF

EXTRA_CHANNELS = {                     # Extra channels to save and quantify
    "C1": 1,                           # Example: "gammaH2AX": 2
}

# ---------- TIFF calibration fallback ----------
DEFAULT_XY_UM = 0.207                  # XY pixel size if TIFF metadata is missing
DEFAULT_Z_UM = 0.2                   # Z spacing if TIFF metadata is missing

# ---------- Manual cleaning and morphometry ----------
REMOVE_BORDER_LABELS = True            # Remove nuclei touching any Z/Y/X image border
MIN_VOLUME_VOX = 500                   # Ignore nuclei smaller than this voxel count
Z_MARGIN = 0                           # Extra Z slices included around each crop
XY_MARGIN = 1                          # Extra Y/X pixels included around each crop
PAD_Z = 1                              # Z padding for marching-cubes surface measurement
PAD_XY = 1                             # Y/X padding for marching-cubes surface measurement

# ---------- Z-slice intensity ----------
CALCULATE_INTENSITY = True             # Add per-nucleus intensity metrics to CSV
INTENSITY_Z_SLICES = {                 # Relative Z positions within each nucleus crop
    "Z25": 0.25,
    "Z50": 0.50,
    "Z75": 0.75,
}
MEASURE_MAX_PROJECTION = True           # Measure mean and sum on a Z-maximum projection
CALCULATE_Z_SLICE_MEAN = True           # Average Z25/Z50/Z75 values

# ---------- Wrinkling Index ----------
Z_PROJECTION_FRACTIONS = [1.0]          # 1.0 = full Z; 0.5 = central 50%
MASK_SHRINK_FACTORS = [0.9]             # Shrink the 2D mask before WI calculation
CANNY_SIGMAS = [1.5]                    # Canny Gaussian sigma
THRESHOLD_MULTIPLIERS = [1.5]           # k in threshold = (mean - k * SD) / reg
THRESHOLD_REG_FACTORS = [0.75]          # Divisor in adaptive threshold formula
CONTRAST_SATURATION = 0.35              # Percent clipped from each intensity tail
# XY pixel size for WI is read automatically from TIFF metadata.
# DEFAULT_XY_UM is used only when the metadata are missing.
FERET_MIN_UM = 2.5                      # Minimum Feret diameter retained as a wrinkle
RIM_EXCLUDE_PX = 4                      # Nuclear rim excluded before final filtering
CANNY_HIGH_RATIO = 1.5                  # High Canny threshold = low threshold * ratio

# ---------- Metadata extraction ----------
CELL_LINE_MAP = {
    "R1": "RPE1",
    "M10": "MCF10A",
    "U2": "U2OS",
    "Hela": "HeLa",
    "MC12": "MC12",
}

REPLICATE_MAP = {
    "Rep1": "1",
    "Rep2": "2",
    "Rep3": "3",
    "Rep4": "4",
}

CONDITION_MAP = {
    "DMSO": "DMSO",
    "LatB": "LatB",
    "Noc": "Noc",
    "LatBNoc": "LatBNoc",
    "ETP": "ETP",
    "Tryp": "Trypsin",
}

DENSITY_MAP = {
    "hi": "high",
    "me": "medium",
    "lo": "low",
}

# ---------- Outputs ----------
OUTPUT_FOLDER = "manualclean_crops_with_rescale"  # Main output folder
WI_OUTPUT_FOLDER = "Wrinkle_Resultsfer2.5rim4"    # WI result folder inside main output
SAVE_CROPS = True                                 # Save ZCYX per-nucleus crops
SAVE_Z_PROJECTIONS = True                         # Save CYX maximum projections
SAVE_EDGE_TIFFS = True                            # Save 5-channel WI diagnostic TIFFs


# =============================================================================
# INPUT / OUTPUT
# =============================================================================

def select_folders() -> tuple[Path, Path]:
    root = Tk()
    root.withdraw()

    image_folder = filedialog.askdirectory(
        title="Select folder containing ZCYX TIFF images"
    )
    mask_folder = filedialog.askdirectory(
        title="Select folder containing Cellpose masks"
    )

    root.destroy()

    if not image_folder or not mask_folder:
        raise SystemExit("Image folder and mask folder are required.")

    return Path(image_folder), Path(mask_folder)


def find_images(folder: Path) -> list[Path]:
    return sorted(
        path
        for path in folder.iterdir()
        if path.suffix.lower() in {".tif", ".tiff"}
        and "_mask" not in path.stem.lower()
        and "_nuc" not in path.stem.lower()
    )


def read_voxel_size_um(path: Path) -> tuple[float, float]:
    with tiff.TiffFile(path) as tif:
        metadata = tif.imagej_metadata or {}

    return (
        float(metadata.get("pixel_width", DEFAULT_XY_UM)),
        float(metadata.get("spacing", DEFAULT_Z_UM)),
    )


def write_tiff(
    path: Path,
    array: np.ndarray,
    axes: str,
    xy_um: float,
    z_um: float | None = None,
) -> None:
    metadata = {
        "axes": axes,
        "unit": "um",
        "pixel_width": xy_um,
        "pixel_height": xy_um,
    }

    if "Z" in axes and z_um is not None:
        metadata["spacing"] = z_um

    tiff.imwrite(
        path,
        array,
        imagej=True,
        metadata=metadata,
    )


def safe_uint16(array: np.ndarray) -> np.ndarray:
    if not np.issubdtype(array.dtype, np.floating):
        return np.clip(array, 0, 65535).astype(np.uint16)

    low = float(np.nanmin(array))
    high = float(np.nanmax(array))

    if high <= low:
        return np.zeros(array.shape, dtype=np.uint16)

    return np.clip(
        (array - low) / (high - low) * 65535,
        0,
        65535,
    ).astype(np.uint16)


# =============================================================================
# METADATA
# =============================================================================

def keyword_match(text: str, mapping: dict[str, str]) -> str:
    text_lower = text.lower()

    for key, value in mapping.items():
        if re.search(
            rf"(^|[_\-\s]){re.escape(key.lower())}($|[_\-\s])",
            text_lower,
        ):
            return value

    return "unknown"


def extract_metadata(filename: str) -> dict:
    return {
        "cell_line": keyword_match(filename, CELL_LINE_MAP),
        "replicate": keyword_match(filename, REPLICATE_MAP),
        "treatment": keyword_match(filename, CONDITION_MAP),
        "density": keyword_match(filename, DENSITY_MAP),
    }


# =============================================================================
# MASK AND MORPHOMETRY HELPERS
# =============================================================================

def remove_labels_touching_border_3d(mask: np.ndarray) -> np.ndarray:
    border_labels = np.unique(
        np.concatenate(
            [
                mask[0].ravel(),
                mask[-1].ravel(),
                mask[:, 0].ravel(),
                mask[:, -1].ravel(),
                mask[:, :, 0].ravel(),
                mask[:, :, -1].ravel(),
            ]
        )
    )
    border_labels = border_labels[border_labels != 0]

    cleaned = mask.copy()

    if border_labels.size:
        cleaned[np.isin(cleaned, border_labels)] = 0

    return cleaned


def pad_mask(mask: np.ndarray) -> np.ndarray:
    return np.pad(
        mask,
        (
            (PAD_Z, PAD_Z),
            (PAD_XY, PAD_XY),
            (PAD_XY, PAD_XY),
        ),
        mode="constant",
    )


def surface_area(mask: np.ndarray, spacing: tuple[float, float, float]) -> float:
    if not mask.any():
        return np.nan

    try:
        vertices, faces, _, _ = measure.marching_cubes(
            mask,
            0.5,
            spacing=spacing,
        )
    except (ValueError, RuntimeError):
        return np.nan

    triangles = vertices[faces]
    cross_products = np.cross(
        triangles[:, 1] - triangles[:, 0],
        triangles[:, 2] - triangles[:, 0],
    )

    return float(
        0.5 * np.linalg.norm(cross_products, axis=1).sum()
    )


def sphericity(volume_um3: float, surface_um2: float) -> float:
    if (
        volume_um3 <= 0
        or not np.isfinite(surface_um2)
        or surface_um2 <= 0
    ):
        return np.nan

    return float(
        np.pi ** (1 / 3)
        * (6 * volume_um3) ** (2 / 3)
        / surface_um2
    )


def excess_surface_area(
    surface_um2: float,
    volume_um3: float,
) -> tuple[float, float]:
    if (
        volume_um3 <= 0
        or not np.isfinite(surface_um2)
        or surface_um2 <= 0
    ):
        return np.nan, np.nan

    sphere_surface_um2 = (
        36 * np.pi * volume_um3**2
    ) ** (1 / 3)

    ratio = surface_um2 / sphere_surface_um2
    return float(ratio), float(ratio - 1)


def rescale_z_isotropic(
    mask: np.ndarray,
    z_um: float,
    xy_um: float,
) -> np.ndarray:
    new_z = int(
        round(mask.shape[0] * z_um / xy_um)
    )

    if new_z <= 1 or new_z == mask.shape[0]:
        return mask.astype(np.uint8)

    return resize(
        mask,
        (
            new_z,
            mask.shape[1],
            mask.shape[2],
        ),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    ).astype(np.uint8)


def calculate_morphometry(
    mask: np.ndarray,
    z_um: float,
    xy_um: float,
) -> dict:
    volume_vox = int(mask.sum())
    anisotropic_spacing = (z_um, xy_um, xy_um)

    volume_um3 = volume_vox * z_um * xy_um**2
    surface_um2 = surface_area(
        pad_mask(mask),
        anisotropic_spacing,
    )

    mask_rescaled = rescale_z_isotropic(
        mask,
        z_um,
        xy_um,
    )
    volume_vox_rescaled = int(mask_rescaled.sum())
    volume_um3_rescaled = (
        volume_vox_rescaled * xy_um**3
    )
    surface_um2_rescaled = surface_area(
        pad_mask(mask_rescaled),
        (xy_um, xy_um, xy_um),
    )

    esa_ratio, esa_excess = excess_surface_area(
        surface_um2_rescaled,
        volume_um3_rescaled,
    )

    return {
        "volume_vox": volume_vox,
        "volume_um3_before": volume_um3,
        "surface_um2_before": surface_um2,
        "sphericity_before": sphericity(
            volume_um3,
            surface_um2,
        ),
        "height_um_before": mask.shape[0] * z_um,
        "volume_um3_rescale": volume_um3_rescaled,
        "surface_um2_rescale": surface_um2_rescaled,
        "sphericity_rescale": sphericity(
            volume_um3_rescaled,
            surface_um2_rescaled,
        ),
        "height_um_rescale": mask_rescaled.shape[0] * xy_um,
        "ESA_ratio_rescale": esa_ratio,
        "ESA_excess_rescale": esa_excess,
    }


# =============================================================================
# Z-SLICE INTENSITY
# =============================================================================

def calculate_channel_intensity(
    image_crop: np.ndarray,
    nucleus_mask: np.ndarray,
) -> dict:
    if not CALCULATE_INTENSITY:
        return {}

    metrics = {}
    z_count = image_crop.shape[0]

    for channel_name, channel_index in EXTRA_CHANNELS.items():
        channel_stack = image_crop[:, channel_index]
        slice_means = []
        slice_sums = []

        if MEASURE_MAX_PROJECTION:
            mask_projection = nucleus_mask.max(axis=0)
            intensity_projection = channel_stack.max(axis=0)
            values = intensity_projection[
                mask_projection
            ].astype(float)

            metrics[f"Zindex_Zmax"] = -1
            metrics[f"MaskArea_px_Zmax"] = int(
                mask_projection.sum()
            )
            metrics[f"Mean_{channel_name}_Zmax"] = (
                float(values.mean())
                if values.size
                else np.nan
            )
            metrics[f"Sum_{channel_name}_Zmax"] = (
                float(values.sum())
                if values.size
                else np.nan
            )

        for slice_name, fraction in INTENSITY_Z_SLICES.items():
            z_index = int(
                round(fraction * (z_count - 1))
            )
            mask_slice = nucleus_mask[z_index]
            values = channel_stack[z_index][
                mask_slice
            ].astype(float)

            metrics[f"Zindex_{slice_name}"] = z_index
            metrics[f"MaskArea_px_{slice_name}"] = int(
                mask_slice.sum()
            )
            metrics[f"Mean_{channel_name}_{slice_name}"] = (
                float(values.mean())
                if values.size
                else np.nan
            )
            metrics[f"Sum_{channel_name}_{slice_name}"] = (
                float(values.sum())
                if values.size
                else np.nan
            )

            slice_means.append(
                metrics[f"Mean_{channel_name}_{slice_name}"]
            )
            slice_sums.append(
                metrics[f"Sum_{channel_name}_{slice_name}"]
            )

        if CALCULATE_Z_SLICE_MEAN:
            metrics[f"Mean_{channel_name}_Zmean"] = float(
                np.nanmean(slice_means)
            )
            metrics[f"Sum_{channel_name}_Zmean"] = float(
                np.nanmean(slice_sums)
            )

    return metrics


# =============================================================================
# WRINKLING INDEX
# =============================================================================

def middle_projection(
    stack: np.ndarray,
    fraction: float,
) -> np.ndarray:
    z_count = stack.shape[0]
    half = fraction / 2

    start = int(
        z_count * (0.5 - half)
    )
    end = int(
        z_count * (0.5 + half)
    )

    return (
        stack.max(axis=0)
        if end <= start
        else stack[start:end].max(axis=0)
    )


def shrink_mask(
    mask: np.ndarray,
    factor: float,
) -> np.ndarray:
    height, width = mask.shape

    affine = transform.AffineTransform(
        scale=(factor, factor),
        translation=(
            width * (1 - factor) / 2,
            height * (1 - factor) / 2,
        ),
    )

    return (
        transform.warp(
            mask.astype(float),
            affine.inverse,
            order=0,
            preserve_range=True,
        )
        > 0.5
    )


def normalize_contrast(
    image: np.ndarray,
) -> np.ndarray:
    low, high = np.percentile(
        image,
        (
            CONTRAST_SATURATION,
            100 - CONTRAST_SATURATION,
        ),
    )

    if high <= low:
        return np.zeros_like(
            image,
            dtype=np.float32,
        )

    return exposure.rescale_intensity(
        image,
        in_range=(low, high),
        out_range=(0, 1),
    ).astype(np.float32)


def filter_edges_by_feret(
    edges: np.ndarray,
    xy_um: float,
) -> np.ndarray:
    labels = measure.label(
        edges,
        connectivity=2,
    )
    filtered = np.zeros_like(
        edges,
        dtype=bool,
    )

    for region in measure.regionprops(labels):
        feret_um = (
            region.feret_diameter_max
            * xy_um
        )

        if feret_um >= FERET_MIN_UM:
            filtered[labels == region.label] = True

    return filtered


def calculate_wrinkling(
    lamin_stack: np.ndarray,
    nucleus_mask: np.ndarray,
    edge_root: Path,
    crop_name: str,
    xy_um: float,
) -> list[dict]:
    rows = []

    full_z_lamin = normalize_contrast(
        lamin_stack.max(axis=0)
    )

    parameter_sets = product(
        Z_PROJECTION_FRACTIONS,
        MASK_SHRINK_FACTORS,
        CANNY_SIGMAS,
        THRESHOLD_MULTIPLIERS,
        THRESHOLD_REG_FACTORS,
    )

    for (
        z_fraction,
        mask_factor,
        canny_sigma,
        threshold_multiplier,
        reg_factor,
    ) in parameter_sets:
        lamin_raw = middle_projection(
            lamin_stack,
            z_fraction,
        )
        mask_2d = middle_projection(
            nucleus_mask,
            z_fraction,
        )
        mask_shrunk = shrink_mask(
            mask_2d,
            mask_factor,
        )
        lamin_normalized = normalize_contrast(
            lamin_raw
        )

        raw_values = lamin_raw[
            mask_shrunk
        ]
        normalized_values = lamin_normalized[
            mask_shrunk
        ]

        raw_stats = {
            "RawMean": (
                float(raw_values.mean())
                if raw_values.size
                else np.nan
            ),
            "RawMedian": (
                float(np.median(raw_values))
                if raw_values.size
                else np.nan
            ),
            "RawStd": (
                float(raw_values.std())
                if raw_values.size
                else np.nan
            ),
            "RawMin": (
                float(raw_values.min())
                if raw_values.size
                else np.nan
            ),
            "RawMax": (
                float(raw_values.max())
                if raw_values.size
                else np.nan
            ),
        }

        if normalized_values.size:
            mean_inside = float(
                normalized_values.mean()
            )
            std_inside = float(
                normalized_values.std()
            )
            threshold_raw = (
                mean_inside
                - threshold_multiplier * std_inside
            ) / reg_factor
            threshold_used = max(
                threshold_raw,
                0,
            )
        else:
            mean_inside = 0.0
            std_inside = 0.0
            threshold_raw = 0.0
            threshold_used = 0.0

        edges = feature.canny(
            lamin_normalized,
            sigma=canny_sigma,
            low_threshold=threshold_used,
            high_threshold=(
                threshold_used
                * CANNY_HIGH_RATIO
            ),
        )

        edges_inside = edges & mask_shrunk
        inner_mask = erosion(
            mask_shrunk,
            disk(RIM_EXCLUDE_PX),
        )
        edges_filtered = filter_edges_by_feret(
            edges_inside & inner_mask,
            xy_um,
        )

        denominator = max(
            1,
            int(mask_shrunk.sum()),
        )

        wi_before = (
            edges_inside.sum()
            / denominator
            * 100
        )
        wi_after = (
            edges_filtered.sum()
            / denominator
            * 100
        )

        parameter_name = (
            f"Z{z_fraction}_scale{mask_factor}"
            f"_sig{canny_sigma}"
            f"_k{threshold_multiplier}"
            f"_reg{reg_factor}"
        )

        if SAVE_EDGE_TIFFS:
            parameter_dir = (
                edge_root / parameter_name
            )
            parameter_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            diagnostic_stack = np.stack(
                [
                    full_z_lamin,
                    lamin_normalized,
                    mask_shrunk,
                    edges_inside,
                    edges_filtered,
                ]
            )

            write_tiff(
                parameter_dir
                / f"{crop_name}_{parameter_name}.tif",
                (
                    diagnostic_stack
                    * 255
                ).astype(np.uint8),
                "CYX",
                xy_um,
            )

        rows.append(
            {
                "Z_Fraction": z_fraction,
                "Mask_Scale": mask_factor,
                "Sigma_Canny": canny_sigma,
                "Thr_Multiplier": threshold_multiplier,
                "RegFactor": reg_factor,
                "Feret_Min_um": FERET_MIN_UM,
                "Rim_Exclude_px": RIM_EXCLUDE_PX,
                **raw_stats,
                "Threshold_Raw": threshold_raw,
                "Threshold_Used": threshold_used,
                "Mean_InsideMask": mean_inside,
                "Std_InsideMask": std_inside,
                "Mask_MidZ_Area_px": float(
                    mask_2d.sum()
                ),
                "Mask_Shrunk_Area_px": float(
                    mask_shrunk.sum()
                ),
                "Mask_MidZ_Area_um2": float(
                    mask_2d.sum()
                    * xy_um**2
                ),
                "Mask_Shrunk_Area_um2": float(
                    mask_shrunk.sum()
                    * xy_um**2
                ),
                "Wrinkling_Index_Before(%)": wi_before,
                "Wrinkling_Index_After(%)": wi_after,
            }
        )

    return rows


# =============================================================================
# PER-NUCLEUS ANALYSIS
# =============================================================================

def crop_bounds(
    region: measure._regionprops.RegionProperties,
    shape: tuple[int, int, int],
) -> tuple[int, int, int, int, int, int]:
    z0, y0, x0, z1, y1, x1 = region.bbox

    return (
        max(0, z0 - Z_MARGIN),
        max(0, y0 - XY_MARGIN),
        max(0, x0 - XY_MARGIN),
        min(shape[0], z1 + Z_MARGIN),
        min(shape[1], y1 + XY_MARGIN),
        min(shape[2], x1 + XY_MARGIN),
    )


def save_nucleus_crop(
    image_crop: np.ndarray,
    nucleus_mask: np.ndarray,
    crop_name: str,
    crop_dir: Path,
    xy_um: float,
    z_um: float,
) -> tuple[Path | None, Path | None]:
    if not SAVE_CROPS:
        return None, None

    crop_channels = [
        image_crop[:, HOECHST_CH] * nucleus_mask,
        image_crop[:, LAMIN_CH] * nucleus_mask,
        nucleus_mask.astype(np.uint16),
    ]

    for channel_index in EXTRA_CHANNELS.values():
        crop_channels.append(
            image_crop[:, channel_index] * nucleus_mask
        )

    stack = np.stack(
        crop_channels,
        axis=1,
    )

    stack_path = (
        crop_dir
        / f"{crop_name}_stack_multich.tif"
    )

    write_tiff(
        stack_path,
        safe_uint16(stack),
        "ZCYX",
        xy_um,
        z_um,
    )

    projection_path = None

    if SAVE_Z_PROJECTIONS:
        projection_path = (
            crop_dir
            / f"{crop_name}_zproj_multich.tif"
        )

        write_tiff(
            projection_path,
            safe_uint16(
                stack.max(axis=0)
            ),
            "CYX",
            xy_um,
        )

    return stack_path, projection_path


def analyze_all_nuclei(
    image_path: Path,
    image: np.ndarray,
    clean_mask: np.ndarray,
    xy_um: float,
    z_um: float,
    crop_dir: Path,
    edge_root: Path,
) -> list[dict]:
    rows = []
    metadata = extract_metadata(
        image_path.stem
    )

    for region in measure.regionprops(
        clean_mask
    ):
        bounds = crop_bounds(
            region,
            clean_mask.shape,
        )
        z0, y0, x0, z1, y1, x1 = bounds

        nucleus_mask = (
            clean_mask[
                z0:z1,
                y0:y1,
                x0:x1,
            ]
            == region.label
        )

        if nucleus_mask.sum() < MIN_VOLUME_VOX:
            continue

        image_crop = image[
            z0:z1,
            :,
            y0:y1,
            x0:x1,
        ]

        crop_name = (
            f"{image_path.stem}"
            f"_nuc{region.label:03d}"
        )

        stack_path, projection_path = (
            save_nucleus_crop(
                image_crop,
                nucleus_mask,
                crop_name,
                crop_dir,
                xy_um,
                z_um,
            )
        )

        common_data = {
            "filename": (
                stack_path.name
                if stack_path is not None
                else f"{crop_name}_stack_multich.tif"
            ),
            "base_name": image_path.stem,
            "nucleus_id": int(region.label),
            "label": int(region.label),
            **metadata,
            "xy_um": xy_um,
            "z_um": z_um,
            **calculate_morphometry(
                nucleus_mask,
                z_um,
                xy_um,
            ),
            **calculate_channel_intensity(
                image_crop,
                nucleus_mask,
            ),
            "stack_path": (
                str(stack_path)
                if stack_path is not None
                else ""
            ),
            "zproj_path": (
                str(projection_path)
                if projection_path is not None
                else ""
            ),
        }

        lamin_stack = image_crop[
            :,
            LAMIN_CH,
        ]

        wrinkle_rows = calculate_wrinkling(
            lamin_stack,
            nucleus_mask,
            edge_root,
            crop_name,
            xy_um,
        )

        rows.extend(
            {
                **common_data,
                **wrinkle_row,
            }
            for wrinkle_row in wrinkle_rows
        )

    return rows


# =============================================================================
# MANUAL REVIEW
# =============================================================================

def review_image(
    image_path: Path,
    mask_path: Path,
    output_dir: Path,
    edge_root: Path,
) -> list[dict]:
    image = tiff.imread(
        image_path
    )
    mask = tiff.imread(
        mask_path
    ).astype(np.int32)

    if image.ndim != 4:
        raise ValueError(
            f"{image_path.name}: expected ZCYX, got {image.shape}."
        )

    if mask.ndim != 3:
        raise ValueError(
            f"{mask_path.name}: expected ZYX, got {mask.shape}."
        )

    channel_count = image.shape[1]
    required_channels = {
        HOECHST_CH,
        LAMIN_CH,
        *EXTRA_CHANNELS.values(),
    }

    invalid_channels = [
        channel
        for channel in required_channels
        if not 0 <= channel < channel_count
    ]

    if invalid_channels:
        raise ValueError(
            f"{image_path.name}: invalid channel indices "
            f"{invalid_channels}; C={channel_count}."
        )

    xy_um, z_um = read_voxel_size_um(
        image_path
    )

    if mask.max() <= 1:
        mask = measure.label(
            mask > 0
        ).astype(np.int32)

    if REMOVE_BORDER_LABELS:
        mask = remove_labels_touching_border_3d(
            mask
        )

    mask = measure.label(
        mask > 0
    ).astype(np.int32)

    viewer = napari.Viewer()
    scale = (z_um, xy_um, xy_um)

    viewer.add_image(
        image[:, HOECHST_CH],
        name="Hoechst",
        colormap="gray",
        scale=scale,
        opacity=0.6,
        blending="additive",
    )
    viewer.add_image(
        image[:, LAMIN_CH],
        name="Lamin",
        colormap="gray",
        scale=scale,
        opacity=0.6,
        blending="additive",
    )

    mask_layer = viewer.add_labels(
        mask.astype(np.uint16),
        name="Mask",
        scale=scale,
        opacity=0.5,
    )

    result_rows = []

    @magicgui(
        call_button="Remove Label",
        label_id={
            "min": 1,
            "max": 1_000_000,
            "step": 1,
            "value": 1,
        },
    )
    def remove_label(label_id: int):
        current = np.asarray(
            mask_layer.data
        ).copy()

        voxel_count = int(
            np.count_nonzero(
                current == label_id
            )
        )

        current[
            current == label_id
        ] = 0

        mask_layer.data = current.astype(
            np.uint16
        )
        mask_layer.refresh()

        viewer.status = (
            f"Removed label {label_id} "
            f"({voxel_count} voxels)"
        )

    @magicgui(
        call_button="Relabel Mask"
    )
    def relabel_mask():
        relabeled = measure.label(
            mask_layer.data > 0
        ).astype(np.uint16)

        mask_layer.data = relabeled
        mask_layer.refresh()

        viewer.status = (
            f"Relabeled to "
            f"{int(relabeled.max())} nuclei"
        )

    @magicgui(
        call_button="Measure, Crop, and Analyze"
    )
    def measure_crop_analyze():
        clean_mask = np.asarray(
            mask_layer.data
        ).astype(np.int32)

        cleaned_mask_path = (
            mask_path.parent
            / f"{image_path.stem}_mask_manualclean.tif"
        )

        write_tiff(
            cleaned_mask_path,
            clean_mask.astype(np.uint16),
            "ZYX",
            xy_um,
            z_um,
        )

        image_rows = analyze_all_nuclei(
            image_path,
            image,
            clean_mask,
            xy_um,
            z_um,
            output_dir,
            edge_root,
        )

        result_rows.extend(
            image_rows
        )

        if image_rows:
            image_csv = (
                output_dir
                / f"{image_path.stem}"
                f"_morphometry_manualclean.csv"
            )

            pd.DataFrame(
                image_rows
            ).to_csv(
                image_csv,
                index=False,
            )

            viewer.status = (
                f"Saved {len(image_rows)} analysis rows"
            )
        else:
            viewer.status = (
                "No valid nuclei were saved"
            )

        viewer.close()

    viewer.window.add_dock_widget(
        remove_label,
        area="right",
    )
    viewer.window.add_dock_widget(
        relabel_mask,
        area="right",
    )
    viewer.window.add_dock_widget(
        measure_crop_analyze,
        area="right",
    )

    viewer.show(block=True)
    viewer.close()
    del viewer

    return result_rows


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    image_dir, mask_dir = select_folders()

    output_dir = (
        image_dir / OUTPUT_FOLDER
    )
    edge_root = (
        output_dir / WI_OUTPUT_FOLDER
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    edge_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_paths = find_images(
        image_dir
    )

    if not image_paths:
        raise FileNotFoundError(
            f"No TIFF images were found in {image_dir}."
        )

    all_rows = []

    for index, image_path in enumerate(
        image_paths,
        start=1,
    ):
        mask_path = (
            mask_dir
            / f"{image_path.stem}"
            f"_cellpose3d_mask.tif"
        )

        if not mask_path.exists():
            print(
                f"[{index}/{len(image_paths)}] "
                f"Skipping {image_path.name}: "
                "Cellpose mask not found."
            )
            continue

        print(
            f"[{index}/{len(image_paths)}] "
            f"Reviewing {image_path.name}"
        )

        image_rows = review_image(
            image_path,
            mask_path,
            output_dir,
            edge_root,
        )

        all_rows.extend(
            image_rows
        )

    if all_rows:
        merged_csv = (
            edge_root
            / "merged_wrinkle_morphology.csv"
        )

        pd.DataFrame(
            all_rows
        ).to_csv(
            merged_csv,
            index=False,
        )

        print(
            f"Saved combined analysis: {merged_csv}"
        )

    print("Manual cleaning and analysis completed.")


if __name__ == "__main__":
    main()